


## **7 High-Impact Projects Included:**

### **1. Revenue Intelligence Dashboard** 
- Daily/Weekly/Monthly revenue trends with YoY growth
- City-wise revenue contribution and market share
- Payment method adoption and preferences
- Revenue per trip and average fare analysis

### **2. Driver Performance & Incentive Optimization** 
- Driver efficiency metrics (trips/day, revenue/driver)
- Idle time and utilization analysis
- Top performer identification for rewards
- Churn prediction indicators

### **3. Rider Behavior & Cohort Analysis** 
- User segmentation (high-value vs casual riders)
- Retention cohorts by signup month
- Lifetime Value (LTV) calculation
- Trip frequency patterns

### **4. Operational Excellence Metrics** 
- Trip completion rate and cancellation analysis
- Peak hours and demand forecasting
- Average trip duration and distance patterns
- No-show and cancellation root cause analysis

### **5. Geographic Expansion Strategy** 
- City-wise performance comparison
- High-demand zones identification
- Supply-demand gap analysis
- Route optimization insights

### **6. Payment Intelligence & Fraud Detection** 
- Payment method preferences by demographics
- Failed transaction patterns
- Anomaly detection in fare amounts
- Cash vs digital payment trends

### **7. Real-time Business Monitoring** 
- Live KPI tracking (trips, revenue, active drivers)
- Hourly performance alerts
- SLA compliance monitoring
- Executive summary dashboard

---

## **Skills Demonstrated:**
-  **Window Functions** (ROW_NUMBER, RANK, LEAD/LAG)
-  **CTEs** (Common Table Expressions)
-  **Cohort Analysis** (Date-based groupings)
-  **Complex Joins** (Self-joins, Multiple table joins)
-  **Aggregations** (GROUP BY, HAVING, ROLLUP)
-  **Date/Time Analysis** (EXTRACT, DATE_TRUNC)
-  **Business Logic** (CASE WHEN, IF statements)
-  **Performance Optimization** (Indexed columns, partition strategies)

---

**Dataset**: 50,000+ Uber trips | **Period**: 2023 Full Year  
**Cities**: San Francisco, New York, Boston, Chicago, Seattle


##  PROJECT 1: Revenue Intelligence Dashboard

### **Business Context:**
As a Data Analyst at Uber, leadership needs to understand:
- Which cities generate the most revenue?
- What's our revenue growth trajectory?
- Which payment methods are most profitable?
- How can we optimize pricing strategies?

### **Key Metrics to Calculate:**
1. Total Revenue by City
2. Month-over-Month (MoM) Growth Rate
3. Average Revenue Per Trip (ARPT)
4. Payment Method Distribution
5. Revenue Concentration (Top 20% cities)

### **Interview Questions This Solves:**
- *"Calculate the revenue contribution of each city and identify the top 3"*
- *"Show MoM growth rate with trend analysis"*
- *"Which payment method has the highest average transaction value?"*
- *"Calculate cumulative revenue and running totals"*

In [0]:
%sql
-- Total Revenue by City with Market Share (%)
-- Demonstrates: Aggregation, Window Functions, Percentage Calculations

WITH city_revenue AS (
  SELECT 
    city,
    COUNT(*) as total_trips,
    ROUND(SUM(fare_amount), 2) as total_revenue,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(AVG(distance_km), 2) as avg_distance
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY city
)
SELECT 
  city,
  total_trips,
  total_revenue,
  avg_fare,
  avg_distance,
  ROUND(total_revenue * 100.0 / SUM(total_revenue) OVER(), 2) as market_share_pct,
  RANK() OVER(ORDER BY total_revenue DESC) as revenue_rank
FROM city_revenue
ORDER BY total_revenue DESC;

-- Business Insight:
-- This query helps identify which cities are revenue drivers
-- and should receive more marketing investment or driver incentives

city,total_trips,total_revenue,avg_fare,avg_distance,market_share_pct,revenue_rank
Boston,7182,115028.2,16.02,7.01,16.92,1
San Francisco,7159,114660.65,16.02,7.04,16.87,2
Seattle,7039,113343.2,16.1,7.08,16.68,3
New York,7036,113175.2,16.09,7.03,16.65,4
Chicago,7091,112791.09,15.91,6.96,16.59,5
Los Angeles,7033,110686.54,15.74,6.94,16.28,6


In [0]:
%sql
-- Monthly Revenue Trend with MoM Growth Rate
-- Demonstrates: Date functions, LAG window function, Growth calculations

WITH monthly_revenue AS (
  SELECT 
    DATE_TRUNC('MONTH', pickup_time) as month,
    ROUND(SUM(fare_amount), 2) as monthly_revenue,
    COUNT(*) as monthly_trips
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY DATE_TRUNC('MONTH', pickup_time)
)
SELECT 
  month,
  monthly_revenue,
  monthly_trips,
  LAG(monthly_revenue) OVER(ORDER BY month) as prev_month_revenue,
  ROUND(
    (monthly_revenue - LAG(monthly_revenue) OVER(ORDER BY month)) * 100.0 / 
    LAG(monthly_revenue) OVER(ORDER BY month), 
    2
  ) as mom_growth_pct,
  SUM(monthly_revenue) OVER(ORDER BY month) as cumulative_revenue
FROM monthly_revenue
ORDER BY month;

-- Business Insight:
-- Negative growth indicates seasonality or market issues
-- Helps in forecasting and resource planning

month,monthly_revenue,monthly_trips,prev_month_revenue,mom_growth_pct,cumulative_revenue
2023-01-01T00:00:00.000Z,606699.99,37966,null,null,606699.99
2023-02-01T00:00:00.000Z,72984.89,4574,606699.99,-87.97,679684.88


In [0]:
%sql
-- Payment Method Performance Analysis
-- Demonstrates: GROUP BY, Aggregations, Conditional Logic

SELECT 
  payment_method,
  COUNT(*) as total_transactions,
  ROUND(SUM(fare_amount), 2) as total_revenue,
  ROUND(AVG(fare_amount), 2) as avg_transaction_value,
  ROUND(AVG(distance_km), 2) as avg_distance,
  ROUND(SUM(fare_amount) * 100.0 / SUM(SUM(fare_amount)) OVER(), 2) as revenue_contribution_pct,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as transaction_share_pct
FROM workspace.default.uber_trips
WHERE status = 'Completed'
GROUP BY payment_method
ORDER BY total_revenue DESC;

-- Business Insight:
-- If digital payments (UPI/Card/Wallet) have higher transaction values,
-- incentivize riders to use them with cashback offers

payment_method,total_transactions,total_revenue,avg_transaction_value,avg_distance,revenue_contribution_pct,transaction_share_pct
Card,10676,171143.2,16.03,7.03,25.18,25.10
Wallet,10644,170020.05,15.97,7.0,25.01,25.02
UPI,10652,169951.07,15.95,6.99,25.0,25.04
Cash,10568,168570.56,15.95,7.0,24.8,24.84


In [0]:
%sql
-- Top 20% Cities Generating 80% Revenue (Pareto Analysis)
-- Demonstrates: NTILE window function, Cumulative calculations

WITH city_revenue AS (
  SELECT 
    city,
    ROUND(SUM(fare_amount), 2) as total_revenue
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY city
),
revenue_with_cumulative AS (
  SELECT 
    city,
    total_revenue,
    SUM(total_revenue) OVER(ORDER BY total_revenue DESC) as cumulative_revenue,
    SUM(total_revenue) OVER() as grand_total_revenue
  FROM city_revenue
)
SELECT 
  city,
  total_revenue,
  cumulative_revenue,
  ROUND(cumulative_revenue * 100.0 / grand_total_revenue, 2) as cumulative_pct
FROM revenue_with_cumulative
WHERE cumulative_revenue <= grand_total_revenue * 0.8
ORDER BY total_revenue DESC;

-- Business Insight:
-- Focus retention efforts on these high-value cities
-- Allocate 80% of marketing budget to top revenue generators

city,total_revenue,cumulative_revenue,cumulative_pct
Boston,115028.2,115028.2,16.92
San Francisco,114660.65,229688.84999999998,33.79
Seattle,113343.2,343032.05,50.47
New York,113175.2,456207.25,67.12


## 🚗 PROJECT 2: Driver Performance & Incentive Optimization

### **Business Context:**
Driver retention is critical for Uber's success. To design effective incentive programs, we need to:
- Identify top-performing drivers for rewards
- Detect underperforming drivers needing training
- Optimize driver allocation across cities
- Predict churn risk based on activity patterns

### **Key Metrics:**
1. Trips per Driver
2. Revenue per Driver
3. Completion Rate
4. Average Trip Duration
5. Driver Activity Segmentation

### **Interview Questions This Solves:**
- *"Find the top 10% of drivers by revenue"*
- *"Calculate driver efficiency metrics"*
- *"Segment drivers into quartiles based on performance"*
- *"Identify drivers at risk of churning (low activity)"*

In [0]:
%sql
-- Top 10% Drivers by Revenue (Reward Program)
-- Demonstrates: NTILE, PERCENTILE, Window Functions

WITH driver_metrics AS (
  SELECT 
    driver_id,
    COUNT(*) as total_trips,
    ROUND(SUM(fare_amount), 2) as total_revenue,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(SUM(distance_km), 2) as total_distance_covered,
    ROUND(SUM(fare_amount) / COUNT(*), 2) as revenue_per_trip,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) * 100.0 / COUNT(*) as completion_rate_pct
  FROM workspace.default.uber_trips
  GROUP BY driver_id
),
driver_with_decile AS (
  SELECT 
    driver_id,
    total_trips,
    total_revenue,
    avg_fare,
    total_distance_covered,
    revenue_per_trip,
    ROUND(completion_rate_pct, 2) as completion_rate_pct,
    NTILE(10) OVER(ORDER BY total_revenue DESC) as performance_decile
  FROM driver_metrics
)
SELECT 
  driver_id,
  total_trips,
  total_revenue,
  avg_fare,
  total_distance_covered,
  revenue_per_trip,
  completion_rate_pct,
  performance_decile
FROM driver_with_decile
WHERE performance_decile = 1  -- Top 10%
ORDER BY total_revenue DESC
LIMIT 100;

-- Business Insight:
-- These drivers should receive:
-- - Priority access to high-demand zones
-- - Bonus incentives (5-10% extra)
-- - Recognition badges in the app

driver_id,total_trips,total_revenue,avg_fare,total_distance_covered,revenue_per_trip,completion_rate_pct,performance_decile
6599,16,283.02,17.69,121.54,17.69,93.75,1
3925,16,280.9,17.56,125.02,17.56,81.25,1
1027,15,262.37,17.49,108.91,17.49,93.33,1
4718,13,256.77,19.75,108.02,19.75,84.62,1
1877,13,254.05,19.54,106.16,19.54,61.54,1
4343,14,252.66,18.05,99.28,18.05,85.71,1
6243,14,251.27,17.95,103.49,17.95,78.57,1
9998,16,250.28,15.64,108.77,15.64,75.00,1
6171,12,248.38,20.7,106.17,20.7,83.33,1
2526,13,239.56,18.43,100.57,18.43,76.92,1


In [0]:
%sql
-- Driver Segmentation: Champions, Regulars, At-Risk, Churners
-- Demonstrates: CASE WHEN, NTILE, Segmentation Logic

WITH driver_performance AS (
  SELECT 
    driver_id,
    COUNT(*) as total_trips,
    ROUND(SUM(fare_amount), 2) as total_revenue,
    NTILE(4) OVER(ORDER BY COUNT(*) DESC) as trip_quartile
  FROM workspace.default.uber_trips
  GROUP BY driver_id
)
SELECT 
  trip_quartile,
  CASE 
    WHEN trip_quartile = 1 THEN 'Champion Drivers'
    WHEN trip_quartile = 2 THEN 'Regular Drivers'
    WHEN trip_quartile = 3 THEN 'At-Risk Drivers'
    WHEN trip_quartile = 4 THEN 'Churning Drivers'
  END as segment_name,
  COUNT(DISTINCT driver_id) as driver_count,
  ROUND(AVG(total_trips), 2) as avg_trips_per_driver,
  ROUND(AVG(total_revenue), 2) as avg_revenue_per_driver,
  ROUND(SUM(total_revenue), 2) as segment_total_revenue
FROM driver_performance
GROUP BY trip_quartile
ORDER BY trip_quartile;

-- Business Insight:
-- Churning Drivers (Q4): Send re-engagement campaigns
-- At-Risk Drivers (Q3): Offer trip completion bonuses
-- Champions (Q1): Invite to exclusive driver events

trip_quartile,segment_name,driver_count,avg_trips_per_driver,avg_revenue_per_driver,segment_total_revenue
1,Champion Drivers,2241,8.66,138.51,310399.9
2,Regular Drivers,2241,6.19,98.8,221411.03
3,At-Risk Drivers,2241,4.62,74.31,166526.23
4,Churning Drivers,2240,2.84,44.85,100473.11


In [0]:
%sql
-- Driver Efficiency: Average Trips per Day
-- Demonstrates: Date functions, Complex aggregations

WITH driver_daily_trips AS (
  SELECT 
    driver_id,
    DATE(pickup_time) as trip_date,
    COUNT(*) as daily_trips,
    ROUND(SUM(fare_amount), 2) as daily_revenue
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY driver_id, DATE(pickup_time)
)
SELECT 
  driver_id,
  COUNT(DISTINCT trip_date) as active_days,
  SUM(daily_trips) as total_trips,
  ROUND(AVG(daily_trips), 2) as avg_trips_per_day,
  ROUND(AVG(daily_revenue), 2) as avg_revenue_per_day,
  MAX(daily_trips) as max_trips_in_a_day,
  CASE 
    WHEN AVG(daily_trips) >= 20 THEN 'Highly Efficient'
    WHEN AVG(daily_trips) >= 10 THEN 'Moderately Efficient'
    ELSE 'Needs Improvement'
  END as efficiency_category
FROM driver_daily_trips
GROUP BY driver_id
HAVING COUNT(DISTINCT trip_date) >= 30  -- Active at least 30 days
ORDER BY avg_trips_per_day DESC
LIMIT 100;

-- Business Insight:
-- Drivers with < 10 trips/day may need:
-- - Better zone recommendations
-- - Peak hour alerts
-- - Training on efficient routing

driver_id,active_days,total_trips,avg_trips_per_day,avg_revenue_per_day,max_trips_in_a_day,efficiency_category


## 👥 PROJECT 3: Rider Behavior & Cohort Analysis

### **Business Context:**
Understanding rider behavior is key to retention and growth. We need to:
- Segment riders by trip frequency and value
- Calculate Customer Lifetime Value (CLV/LTV)
- Analyze cohort retention patterns
- Identify high-value riders for loyalty programs

### **Key Metrics:**
1. Rider Segmentation (High/Medium/Low Value)
2. Lifetime Value (LTV)
3. Trip Frequency Distribution
4. Cohort Retention Rate
5. Churn Prediction Signals

### **Interview Questions This Solves:**
- *"Segment users into high, medium, and low-value customers"*
- *"Calculate the average lifetime value of a rider"*
- *"Show month-over-month retention for user cohorts"*
- *"Identify riders who haven't taken a trip in 30+ days"*

In [0]:
%sql
-- RFM Analysis: Recency, Frequency, Monetary Value
-- Demonstrates: Complex CASE statements, Multi-dimensional segmentation

WITH rider_rfm AS (
  SELECT 
    rider_id,
    DATEDIFF(CURRENT_DATE(), MAX(DATE(pickup_time))) as days_since_last_trip,  -- Recency
    COUNT(*) as total_trips,  -- Frequency
    ROUND(SUM(fare_amount), 2) as total_spent  -- Monetary
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY rider_id
),
rfm_scores AS (
  SELECT 
    rider_id,
    days_since_last_trip,
    total_trips,
    total_spent,
    NTILE(5) OVER(ORDER BY days_since_last_trip ASC) as recency_score,
    NTILE(5) OVER(ORDER BY total_trips DESC) as frequency_score,
    NTILE(5) OVER(ORDER BY total_spent DESC) as monetary_score
  FROM rider_rfm
)
SELECT 
  rider_id,
  days_since_last_trip,
  total_trips,
  total_spent,
  recency_score,
  frequency_score,
  monetary_score,
  (recency_score + frequency_score + monetary_score) as rfm_total_score,
  CASE 
    WHEN (recency_score + frequency_score + monetary_score) >= 13 THEN 'Champions'
    WHEN (recency_score + frequency_score + monetary_score) >= 10 THEN 'Loyal Customers'
    WHEN (recency_score + frequency_score + monetary_score) >= 7 THEN 'Potential Loyalists'
    WHEN recency_score <= 2 THEN 'At Risk / Churning'
    ELSE 'Needs Attention'
  END as customer_segment
FROM rfm_scores
ORDER BY rfm_total_score DESC
LIMIT 1000;

-- Business Insight:
-- Champions: VIP treatment, exclusive offers
-- At Risk: Win-back campaigns with 20% off coupons

rider_id,days_since_last_trip,total_trips,total_spent,recency_score,frequency_score,monetary_score,rfm_total_score,customer_segment
46349,1246,1,10.43,5,3,5,13,Champions
59633,1246,1,8.64,5,3,5,13,Champions
33775,1246,1,11.06,5,3,5,13,Champions
22980,1246,1,10.18,5,3,5,13,Champions
47768,1246,1,9.43,5,3,5,13,Champions
69032,1246,1,8.31,5,3,5,13,Champions
67439,1246,1,3.55,5,3,5,13,Champions
39193,1246,1,10.89,5,3,5,13,Champions
11058,1246,1,6.55,5,3,5,13,Champions
56858,1246,1,10.16,5,3,5,13,Champions


In [0]:
%sql
-- Customer Lifetime Value (CLV) with Cohort
-- Demonstrates: AVG, aggregations, cohort grouping

WITH rider_ltv AS (
  SELECT 
    rider_id,
    MIN(DATE(pickup_time)) as first_trip_date,
    MAX(DATE(pickup_time)) as last_trip_date,
    COUNT(*) as lifetime_trips,
    ROUND(SUM(fare_amount), 2) as lifetime_value,
    ROUND(AVG(fare_amount), 2) as avg_trip_value,
    DATEDIFF(MAX(DATE(pickup_time)), MIN(DATE(pickup_time))) as customer_lifespan_days
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY rider_id
)
SELECT 
  rider_id,
  first_trip_date,
  last_trip_date,
  lifetime_trips,
  lifetime_value,
  avg_trip_value,
  customer_lifespan_days,
  ROUND(lifetime_value / NULLIF(customer_lifespan_days, 0) * 30, 2) as monthly_value,
  CASE 
    WHEN lifetime_value >= 500 THEN 'High Value'
    WHEN lifetime_value >= 200 THEN 'Medium Value'
    ELSE 'Low Value'
  END as value_segment
FROM rider_ltv
WHERE customer_lifespan_days > 0
ORDER BY lifetime_value DESC
LIMIT 1000;

-- Business Insight:
-- High Value riders (>$500 LTV): Offer subscription plans
-- Target acquisition cost should be < 30% of expected LTV

rider_id,first_trip_date,last_trip_date,lifetime_trips,lifetime_value,avg_trip_value,customer_lifespan_days,monthly_value,value_segment
59258,2023-01-02,2023-02-02,6,117.92,19.65,31,114.12,Low Value
32370,2023-01-06,2023-01-26,5,106.52,21.3,20,159.78,Low Value
17566,2023-01-11,2023-01-29,5,97.03,19.41,18,161.72,Low Value
41545,2023-01-05,2023-02-02,4,96.79,24.2,28,103.7,Low Value
50740,2023-01-15,2023-01-29,4,96.02,24.01,14,205.76,Low Value
40500,2023-01-10,2023-02-02,4,95.31,23.83,23,124.32,Low Value
34236,2023-01-03,2023-02-04,5,93.54,18.71,32,87.69,Low Value
49680,2023-01-01,2023-01-26,5,92.68,18.54,25,111.22,Low Value
52132,2023-01-02,2023-01-19,4,90.24,22.56,17,159.25,Low Value
38537,2023-01-03,2023-01-24,4,90.24,22.56,21,128.91,Low Value


In [0]:
%sql
-- Monthly Cohort Retention Rate
-- Demonstrates: Self-join, Cohort analysis, Retention calculation

WITH first_trip AS (
  SELECT 
    rider_id,
    DATE_TRUNC('MONTH', MIN(pickup_time)) as cohort_month
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY rider_id
),
user_activity AS (
  SELECT DISTINCT
    rider_id,
    DATE_TRUNC('MONTH', pickup_time) as activity_month
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
)
SELECT 
  f.cohort_month,
  COUNT(DISTINCT f.rider_id) as cohort_size,
  COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 0 THEN a.rider_id END) as month_0,
  COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 1 THEN a.rider_id END) as month_1,
  COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 2 THEN a.rider_id END) as month_2,
  COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 3 THEN a.rider_id END) as month_3,
  ROUND(COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 1 THEN a.rider_id END) * 100.0 / COUNT(DISTINCT f.rider_id), 2) as month_1_retention_pct,
  ROUND(COUNT(DISTINCT CASE WHEN DATEDIFF(MONTH, f.cohort_month, a.activity_month) = 3 THEN a.rider_id END) * 100.0 / COUNT(DISTINCT f.rider_id), 2) as month_3_retention_pct
FROM first_trip f
LEFT JOIN user_activity a ON f.rider_id = a.rider_id
GROUP BY f.cohort_month
ORDER BY f.cohort_month;

-- Business Insight:
-- If Month 1 retention < 40%, improve onboarding experience
-- Target: 30-40% retention after 3 months

cohort_month,cohort_size,month_0,month_1,month_2,month_3,month_1_retention_pct,month_3_retention_pct
2023-01-01T00:00:00.000Z,30961,30961,1541,0,0,4.98,0.00
2023-02-01T00:00:00.000Z,2929,2929,0,0,0,0.00,0.00


## ⚙️ PROJECT 4: Operational Excellence Metrics

### **Business Context:**
Operational efficiency directly impacts profitability. Key questions:
- What's our trip completion rate?
- When do cancellations/no-shows peak?
- How can we reduce idle time?
- What's the average trip duration by city?

### **Key Metrics:**
1. Trip Completion Rate
2. Cancellation & No-Show Analysis
3. Peak Hour Demand Patterns
4. Average Trip Duration
5. Supply-Demand Mismatch

### **Interview Questions This Solves:**
- *"Calculate the overall completion rate and breakdown by city"*
- *"Identify peak hours with highest trip volume"*
- *"What percentage of trips result in no-shows?"*
- *"Calculate average wait time and trip duration"*

In [0]:
%sql
-- Trip Completion Rate and Failure Analysis
-- Demonstrates: CASE aggregations, Percentage calculations

SELECT 
  city,
  COUNT(*) as total_trips,
  COUNT(CASE WHEN status = 'Completed' THEN 1 END) as completed_trips,
  COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) as cancelled_trips,
  COUNT(CASE WHEN status = 'No-Show' THEN 1 END) as no_show_trips,
  ROUND(COUNT(CASE WHEN status = 'Completed' THEN 1 END) * 100.0 / COUNT(*), 2) as completion_rate_pct,
  ROUND(COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) * 100.0 / COUNT(*), 2) as cancellation_rate_pct,
  ROUND(COUNT(CASE WHEN status = 'No-Show' THEN 1 END) * 100.0 / COUNT(*), 2) as no_show_rate_pct
FROM workspace.default.uber_trips
GROUP BY city
ORDER BY completion_rate_pct DESC;

-- Business Insight:
-- Cities with completion rate < 80% need driver training
-- High no-show rates indicate rider behavior issues

city,total_trips,completed_trips,cancelled_trips,no_show_trips,completion_rate_pct,cancellation_rate_pct,no_show_rate_pct
Seattle,8238,7039,806,393,85.45,9.78,4.77
New York,8247,7036,809,402,85.32,9.81,4.87
San Francisco,8401,7159,811,431,85.22,9.65,5.13
Chicago,8345,7091,839,415,84.97,10.05,4.97
Boston,8454,7182,873,399,84.95,10.33,4.72
Los Angeles,8315,7033,846,436,84.58,10.17,5.24


In [0]:
%sql
-- Peak Hour Analysis - Demand Forecasting
-- Demonstrates: EXTRACT, Hour-based grouping, Demand patterns

SELECT 
  EXTRACT(HOUR FROM pickup_time) as hour_of_day,
  COUNT(*) as total_trips,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  ROUND(AVG(distance_km), 2) as avg_distance,
  COUNT(CASE WHEN status = 'Completed' THEN 1 END) as completed_trips,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_of_daily_trips,
  CASE 
    WHEN EXTRACT(HOUR FROM pickup_time) BETWEEN 7 AND 9 THEN 'Morning Peak'
    WHEN EXTRACT(HOUR FROM pickup_time) BETWEEN 17 AND 20 THEN 'Evening Peak'
    WHEN EXTRACT(HOUR FROM pickup_time) BETWEEN 22 AND 23 THEN 'Night Shift'
    ELSE 'Off-Peak'
  END as time_category
FROM workspace.default.uber_trips
GROUP BY EXTRACT(HOUR FROM pickup_time)
ORDER BY hour_of_day;

-- Business Insight:
-- Deploy more drivers during peak hours (7-9 AM, 5-8 PM)
-- Implement surge pricing during high-demand periods

hour_of_day,total_trips,avg_fare,avg_distance,completed_trips,pct_of_daily_trips,time_category
0,2100,15.77,6.91,1821,4.20,Off-Peak
1,2100,15.92,6.96,1772,4.20,Off-Peak
2,2100,15.78,6.87,1803,4.20,Off-Peak
3,2100,16.16,7.12,1776,4.20,Off-Peak
4,2100,15.96,7.01,1775,4.20,Off-Peak
5,2100,15.97,7.02,1775,4.20,Off-Peak
6,2100,16.16,7.06,1784,4.20,Off-Peak
7,2100,15.87,6.92,1756,4.20,Morning Peak
8,2100,16.12,7.07,1810,4.20,Morning Peak
9,2100,16.09,7.01,1802,4.20,Morning Peak


In [0]:
%sql
-- Trip Duration and Distance Analysis
-- Demonstrates: TIMESTAMPDIFF, Time calculations, Percentile functions

WITH trip_duration AS (
  SELECT 
    trip_id,
    city,
    status,
    distance_km,
    fare_amount,
    pickup_time,
    drop_time,
    ROUND(TIMESTAMPDIFF(MINUTE, pickup_time, drop_time), 2) as duration_minutes
  FROM workspace.default.uber_trips
  WHERE status = 'Completed' AND drop_time IS NOT NULL
)
SELECT 
  city,
  COUNT(*) as total_trips,
  ROUND(AVG(duration_minutes), 2) as avg_duration_min,
  ROUND(AVG(distance_km), 2) as avg_distance_km,
  ROUND(AVG(distance_km / NULLIF(duration_minutes, 0) * 60), 2) as avg_speed_kmph,
  ROUND(PERCENTILE(duration_minutes, 0.5), 2) as median_duration_min,
  ROUND(PERCENTILE(distance_km, 0.5), 2) as median_distance_km,
  MAX(duration_minutes) as max_duration_min
FROM trip_duration
GROUP BY city
ORDER BY avg_duration_min DESC;

-- Business Insight:
-- Cities with avg_speed < 20 kmph indicate traffic congestion
-- Consider alternate routing or bike services

city,total_trips,avg_duration_min,avg_distance_km,avg_speed_kmph,median_duration_min,median_distance_km,max_duration_min
Seattle,7039,20.76,7.08,20.7,21.0,7.06,54
San Francisco,7159,20.61,7.04,20.69,21.0,7.0,52
New York,7036,20.59,7.03,20.69,20.0,6.95,55
Boston,7182,20.53,7.01,20.7,20.5,7.0,55
Chicago,7091,20.38,6.96,20.7,20.0,6.93,54
Los Angeles,7033,20.32,6.94,20.71,20.0,6.92,54


In [0]:
%sql
-- Day of Week Performance Analysis
-- Demonstrates: Date functions, Weekly patterns

SELECT 
  DAYOFWEEK(pickup_time) as day_number,
  CASE DAYOFWEEK(pickup_time)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END as day_name,
  COUNT(*) as total_trips,
  ROUND(SUM(fare_amount), 2) as total_revenue,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  COUNT(CASE WHEN status = 'Completed' THEN 1 END) as completed_trips,
  ROUND(COUNT(CASE WHEN status = 'Completed' THEN 1 END) * 100.0 / COUNT(*), 2) as completion_rate_pct
FROM workspace.default.uber_trips
GROUP BY DAYOFWEEK(pickup_time)
ORDER BY day_number;

-- Business Insight:
-- Weekends (Sat/Sun) typically have different demand patterns
-- Adjust driver availability accordingly

day_number,day_name,total_trips,total_revenue,avg_fare,completed_trips,completion_rate_pct
1,Sunday,7200,114985.94,15.97,6120,85.00
2,Monday,7200,116032.1,16.12,6127,85.10
3,Tuesday,7200,115462.03,16.04,6091,84.60
4,Wednesday,7200,115064.37,15.98,6151,85.43
5,Thursday,7200,114586.74,15.91,6114,84.92
6,Friday,7200,114947.18,15.96,6120,85.00
7,Saturday,6800,107731.91,15.84,5817,85.54


## 🗺️ PROJECT 5: Geographic Expansion Strategy

### **Business Context:**
To expand into new markets, leadership needs data-driven insights:
- Which cities have the highest growth potential?
- Where are the high-demand zones?
- How does performance vary across geographies?
- Should we invest more in existing cities or explore new ones?

### **Key Metrics:**
1. City-wise CAGR (Compound Annual Growth Rate)
2. Revenue per Square Kilometer
3. Supply-Demand Ratio
4. Market Penetration Rate
5. Geographic Hotspots

### **Interview Questions This Solves:**
- *"Which city has shown the highest growth in the last quarter?"*
- *"Compare performance metrics across all cities"*
- *"Identify underserved areas with high demand"*
- *"Calculate market concentration and diversification"*

In [0]:
%sql
-- Comprehensive City Performance Scorecard
-- Demonstrates: Complex aggregations, Multiple metrics, Ranking

WITH city_metrics AS (
  SELECT 
    city,
    COUNT(*) as total_trips,
    COUNT(DISTINCT driver_id) as active_drivers,
    COUNT(DISTINCT rider_id) as active_riders,
    ROUND(SUM(fare_amount), 2) as total_revenue,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(AVG(distance_km), 2) as avg_distance,
    ROUND(SUM(distance_km), 2) as total_distance_covered,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) * 100.0 / COUNT(*) as completion_rate
  FROM workspace.default.uber_trips
  GROUP BY city
)
SELECT 
  city,
  total_trips,
  active_drivers,
  active_riders,
  total_revenue,
  avg_fare,
  avg_distance,
  ROUND(total_revenue / active_drivers, 2) as revenue_per_driver,
  ROUND(total_revenue / active_riders, 2) as revenue_per_rider,
  ROUND(total_trips / active_drivers, 2) as trips_per_driver,
  ROUND(completion_rate, 2) as completion_rate_pct,
  RANK() OVER(ORDER BY total_revenue DESC) as revenue_rank,
  RANK() OVER(ORDER BY completion_rate DESC) as quality_rank
FROM city_metrics
ORDER BY total_revenue DESC;

-- Business Insight:
-- Cities with high revenue but low completion rate need operational fixes
-- Focus expansion on high revenue_per_rider markets

city,total_trips,active_drivers,active_riders,total_revenue,avg_fare,avg_distance,revenue_per_driver,revenue_per_rider,trips_per_driver,completion_rate_pct,revenue_rank,quality_rank
Boston,8454,5455,8054,135704.41,16.05,7.02,24.88,16.85,1.55,84.95,1,5
San Francisco,8401,5483,8014,134611.24,16.02,7.03,24.55,16.8,1.53,85.22,2,3
Chicago,8345,5443,7945,132693.51,15.9,6.96,24.38,16.7,1.53,84.97,3,4
New York,8247,5373,7897,132351.73,16.05,7.01,24.63,16.76,1.53,85.32,4,2
Seattle,8238,5340,7873,132293.3,16.06,7.07,24.77,16.8,1.54,85.45,5,1
Los Angeles,8315,5512,7961,131156.08,15.77,6.95,23.79,16.47,1.51,84.58,6,6


In [0]:
%sql
-- Supply-Demand Balance Analysis
-- Demonstrates: Ratio calculations, Gap analysis

WITH city_supply_demand AS (
  SELECT 
    city,
    COUNT(DISTINCT rider_id) as total_riders,
    COUNT(DISTINCT driver_id) as total_drivers,
    COUNT(*) as total_trips,
    ROUND(COUNT(DISTINCT rider_id) * 1.0 / NULLIF(COUNT(DISTINCT driver_id), 0), 2) as rider_to_driver_ratio,
    ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT driver_id), 0), 2) as trips_per_driver
  FROM workspace.default.uber_trips
  GROUP BY city
)
SELECT 
  city,
  total_riders,
  total_drivers,
  total_trips,
  rider_to_driver_ratio,
  trips_per_driver,
  CASE 
    WHEN rider_to_driver_ratio > 15 THEN 'High Demand - Need More Drivers'
    WHEN rider_to_driver_ratio BETWEEN 8 AND 15 THEN 'Balanced'
    ELSE 'Oversupply - Need Marketing'
  END as market_condition
FROM city_supply_demand
ORDER BY rider_to_driver_ratio DESC;

-- Business Insight:
-- Rider:Driver ratio > 15 indicates undersupply
-- Recruit more drivers in high-demand cities

city,total_riders,total_drivers,total_trips,rider_to_driver_ratio,trips_per_driver,market_condition
Boston,8054,5455,8454,1.48,1.55,Oversupply - Need Marketing
Seattle,7873,5340,8238,1.47,1.54,Oversupply - Need Marketing
New York,7897,5373,8247,1.47,1.53,Oversupply - Need Marketing
Chicago,7945,5443,8345,1.46,1.53,Oversupply - Need Marketing
San Francisco,8014,5483,8401,1.46,1.53,Oversupply - Need Marketing
Los Angeles,7961,5512,8315,1.44,1.51,Oversupply - Need Marketing


In [0]:
%sql
-- City-wise Growth Trajectory
-- Demonstrates: LAG function, Growth rate calculation, Time series

WITH monthly_city_performance AS (
  SELECT 
    city,
    DATE_TRUNC('MONTH', pickup_time) as month,
    COUNT(*) as monthly_trips,
    ROUND(SUM(fare_amount), 2) as monthly_revenue
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY city, DATE_TRUNC('MONTH', pickup_time)
)
SELECT 
  city,
  month,
  monthly_trips,
  monthly_revenue,
  LAG(monthly_revenue) OVER(PARTITION BY city ORDER BY month) as prev_month_revenue,
  ROUND(
    (monthly_revenue - LAG(monthly_revenue) OVER(PARTITION BY city ORDER BY month)) * 100.0 / 
    NULLIF(LAG(monthly_revenue) OVER(PARTITION BY city ORDER BY month), 0),
    2
  ) as mom_growth_pct
FROM monthly_city_performance
ORDER BY city, month;

-- Business Insight:
-- Cities with consistent positive growth deserve more investment
-- Declining cities need marketing interventions

city,month,monthly_trips,monthly_revenue,prev_month_revenue,mom_growth_pct
Boston,2023-01-01T00:00:00.000Z,6425,103088.62,null,null
Boston,2023-02-01T00:00:00.000Z,757,11939.58,103088.62,-88.42
Chicago,2023-01-01T00:00:00.000Z,6269,99493.73,null,null
Chicago,2023-02-01T00:00:00.000Z,822,13297.36,99493.73,-86.63
Los Angeles,2023-01-01T00:00:00.000Z,6289,99105.57,null,null
Los Angeles,2023-02-01T00:00:00.000Z,744,11580.97,99105.57,-88.31
New York,2023-01-01T00:00:00.000Z,6306,101426.69,null,null
New York,2023-02-01T00:00:00.000Z,730,11748.51,101426.69,-88.42
San Francisco,2023-01-01T00:00:00.000Z,6381,102191.76,null,null
San Francisco,2023-02-01T00:00:00.000Z,778,12468.89,102191.76,-87.8


## 💳 PROJECT 6: Payment Intelligence & Fraud Detection

### **Business Context:**
Payment data reveals crucial insights about:
- Customer payment preferences
- Potential fraud patterns
- Failed transaction rates
- Digital payment adoption

### **Key Metrics:**
1. Payment Method Distribution
2. Average Transaction Value by Method
3. Failed Payment Rate
4. Anomaly Detection (Unusually High Fares)
5. Cash vs Digital Trends

### **Interview Questions This Solves:**
- *"Which payment method has the lowest transaction failure rate?"*
- *"Detect trips with fare amounts significantly above average"*
- *"Calculate the adoption rate of digital payments over time"*
- *"Identify potential fraudulent transactions"*

In [0]:
%sql
-- Payment Method Adoption Over Time
-- Demonstrates: Time series, Payment trends, Percentage distribution

WITH monthly_payment AS (
  SELECT 
    DATE_TRUNC('MONTH', pickup_time) as month,
    payment_method,
    COUNT(*) as transaction_count,
    ROUND(SUM(fare_amount), 2) as total_amount
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
  GROUP BY DATE_TRUNC('MONTH', pickup_time), payment_method
)
SELECT 
  month,
  payment_method,
  transaction_count,
  total_amount,
  ROUND(transaction_count * 100.0 / SUM(transaction_count) OVER(PARTITION BY month), 2) as pct_of_monthly_transactions,
  SUM(transaction_count) OVER(PARTITION BY payment_method ORDER BY month) as cumulative_transactions
FROM monthly_payment
ORDER BY month, transaction_count DESC;

-- Business Insight:
-- Track digital payment adoption (UPI + Card + Wallet)
-- Goal: 70%+ digital payment adoption

month,payment_method,transaction_count,total_amount,pct_of_monthly_transactions,cumulative_transactions
2023-01-01T00:00:00.000Z,Card,9567,153258.24,25.20,9567
2023-01-01T00:00:00.000Z,UPI,9499,151742.35,25.02,9499
2023-01-01T00:00:00.000Z,Wallet,9485,151362.59,24.98,9485
2023-01-01T00:00:00.000Z,Cash,9415,150336.81,24.80,9415
2023-02-01T00:00:00.000Z,Wallet,1159,18657.46,25.34,10644
2023-02-01T00:00:00.000Z,Cash,1153,18233.75,25.21,10568
2023-02-01T00:00:00.000Z,UPI,1153,18208.72,25.21,10652
2023-02-01T00:00:00.000Z,Card,1109,17884.96,24.25,10676


In [0]:
%sql
-- Anomaly Detection: Suspicious High-Value Trips
-- Demonstrates: Statistical analysis, Z-score, Outlier detection

WITH fare_stats AS (
  SELECT 
    AVG(fare_amount) as avg_fare,
    STDDEV(fare_amount) as stddev_fare
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
),
trips_with_zscore AS (
  SELECT 
    t.trip_id,
    t.driver_id,
    t.rider_id,
    t.city,
    t.distance_km,
    t.fare_amount,
    t.payment_method,
    t.pickup_time,
    ROUND((t.fare_amount - fs.avg_fare) / NULLIF(fs.stddev_fare, 0), 2) as z_score
  FROM workspace.default.uber_trips t
  CROSS JOIN fare_stats fs
  WHERE t.status = 'Completed'
)
SELECT 
  trip_id,
  driver_id,
  rider_id,
  city,
  distance_km,
  fare_amount,
  payment_method,
  pickup_time,
  z_score,
  CASE 
    WHEN z_score > 3 THEN 'High Risk - Investigate'
    WHEN z_score > 2 THEN 'Medium Risk - Review'
    ELSE 'Normal'
  END as fraud_risk_level
FROM trips_with_zscore
WHERE z_score > 2  -- Outliers beyond 2 standard deviations
ORDER BY z_score DESC
LIMIT 100;

-- Business Insight:
-- Trips with z_score > 3 should trigger automatic alerts
-- Review driver behavior for repeated anomalies

trip_id,driver_id,rider_id,city,distance_km,fare_amount,payment_method,pickup_time,z_score,fraud_risk_level
20298,7678,43787,New York,17.7,48.2,Cash,2023-01-15T02:17:00.000Z,5.13,High Risk - Investigate
15667,6593,98176,New York,18.6,47.93,UPI,2023-01-11T21:06:00.000Z,5.09,High Risk - Investigate
25837,2821,97257,Seattle,17.66,47.9,Wallet,2023-01-18T22:36:00.000Z,5.08,High Risk - Investigate
39082,5042,86234,Seattle,17.72,47.35,UPI,2023-01-28T03:21:00.000Z,5.0,High Risk - Investigate
4379,5159,63580,Los Angeles,18.08,46.43,UPI,2023-01-04T00:58:00.000Z,4.85,High Risk - Investigate
4388,9975,29327,Seattle,17.36,44.27,UPI,2023-01-04T01:07:00.000Z,4.51,High Risk - Investigate
29471,9165,18976,Chicago,16.26,43.56,UPI,2023-01-21T11:10:00.000Z,4.39,High Risk - Investigate
38093,6984,18642,Boston,15.97,43.54,UPI,2023-01-27T10:52:00.000Z,4.39,High Risk - Investigate
20216,3100,97927,Seattle,17.01,42.9,Cash,2023-01-15T00:55:00.000Z,4.29,High Risk - Investigate
4653,1746,66522,Los Angeles,17.04,42.88,Wallet,2023-01-04T05:32:00.000Z,4.28,High Risk - Investigate


In [0]:
%sql
-- Cash vs Digital Payment Behavior
-- Demonstrates: Grouping by categories, Trend analysis

WITH payment_category AS (
  SELECT 
    trip_id,
    fare_amount,
    distance_km,
    city,
    DATE_TRUNC('MONTH', pickup_time) as month,
    CASE 
      WHEN payment_method = 'Cash' THEN 'Cash'
      ELSE 'Digital'
    END as payment_category
  FROM workspace.default.uber_trips
  WHERE status = 'Completed'
)
SELECT 
  month,
  payment_category,
  COUNT(*) as trip_count,
  ROUND(SUM(fare_amount), 2) as total_revenue,
  ROUND(AVG(fare_amount), 2) as avg_transaction_value,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY month), 2) as pct_share
FROM payment_category
GROUP BY month, payment_category
ORDER BY month, payment_category;

-- Business Insight:
-- If digital % is increasing, reduce cash handling costs
-- Offer 5-10% cashback on digital payments to accelerate adoption

month,payment_category,trip_count,total_revenue,avg_transaction_value,pct_share
2023-01-01T00:00:00.000Z,Cash,9415,150336.81,15.97,24.80
2023-01-01T00:00:00.000Z,Digital,28551,456363.18,15.98,75.20
2023-02-01T00:00:00.000Z,Cash,1153,18233.75,15.81,25.21
2023-02-01T00:00:00.000Z,Digital,3421,54751.14,16.0,74.79


## 📈 PROJECT 7: Real-time Business Monitoring Dashboard

### **Business Context:**
Executives need real-time visibility into business health:
- What's our performance today vs yesterday?
- Are we meeting revenue targets?
- Which metrics are declining?
- Real-time alerts for anomalies

### **Key Metrics:**
1. Today vs Yesterday Comparison
2. Hourly Revenue Tracking
3. Live Driver Utilization
4. Real-time Conversion Rate
5. SLA Compliance Monitoring

### **Interview Questions This Solves:**
- *"Compare today's performance with last week"*
- *"Calculate running totals for the current month"*
- *"Show hourly revenue with targets"*
- *"Identify declining metrics in real-time"*

In [0]:
%sql
-- Daily Performance Comparison: Today vs Yesterday
-- Demonstrates: Date filtering, YoY/DoD comparisons

WITH today_metrics AS (
  SELECT 
    COUNT(*) as trips_today,
    ROUND(SUM(fare_amount), 2) as revenue_today,
    COUNT(DISTINCT driver_id) as active_drivers_today,
    COUNT(DISTINCT rider_id) as active_riders_today
  FROM workspace.default.uber_trips
  WHERE DATE(pickup_time) = CURRENT_DATE()
),
yesterday_metrics AS (
  SELECT 
    COUNT(*) as trips_yesterday,
    ROUND(SUM(fare_amount), 2) as revenue_yesterday,
    COUNT(DISTINCT driver_id) as active_drivers_yesterday,
    COUNT(DISTINCT rider_id) as active_riders_yesterday
  FROM workspace.default.uber_trips
  WHERE DATE(pickup_time) = CURRENT_DATE() - INTERVAL 1 DAY
)
SELECT 
  t.trips_today,
  y.trips_yesterday,
  t.trips_today - y.trips_yesterday as trip_change,
  ROUND((t.trips_today - y.trips_yesterday) * 100.0 / NULLIF(y.trips_yesterday, 0), 2) as trip_change_pct,
  t.revenue_today,
  y.revenue_yesterday,
  t.revenue_today - y.revenue_yesterday as revenue_change,
  ROUND((t.revenue_today - y.revenue_yesterday) * 100.0 / NULLIF(y.revenue_yesterday, 0), 2) as revenue_change_pct,
  t.active_drivers_today,
  y.active_drivers_yesterday,
  t.active_riders_today,
  y.active_riders_yesterday
FROM today_metrics t, yesterday_metrics y;

-- Business Insight:
-- Negative day-over-day growth requires immediate action
-- Share this dashboard with leadership every morning

trips_today,trips_yesterday,trip_change,trip_change_pct,revenue_today,revenue_yesterday,revenue_change,revenue_change_pct,active_drivers_today,active_drivers_yesterday,active_riders_today,active_riders_yesterday
0,0,0,null,null,null,null,null,0,0,0,0


In [0]:
%sql
-- Hourly Revenue Tracking for Today
-- Demonstrates: Real-time monitoring, Hourly breakdown

SELECT 
  EXTRACT(HOUR FROM pickup_time) as hour_of_day,
  COUNT(*) as hourly_trips,
  ROUND(SUM(fare_amount), 2) as hourly_revenue,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  SUM(COUNT(*)) OVER(ORDER BY EXTRACT(HOUR FROM pickup_time)) as cumulative_trips,
  ROUND(SUM(SUM(fare_amount)) OVER(ORDER BY EXTRACT(HOUR FROM pickup_time)), 2) as cumulative_revenue
FROM workspace.default.uber_trips
WHERE DATE(pickup_time) = CURRENT_DATE()
  AND status = 'Completed'
GROUP BY EXTRACT(HOUR FROM pickup_time)
ORDER BY hour_of_day;

-- Business Insight:
-- Track progress toward daily revenue targets
-- Alert if hourly revenue drops below threshold

hour_of_day,hourly_trips,hourly_revenue,avg_fare,cumulative_trips,cumulative_revenue


In [0]:
%sql
-- Weekly Executive Summary (Last 7 Days)
-- Demonstrates: Rolling window, Multi-metric summary

WITH last_7_days AS (
  SELECT 
    DATE(pickup_time) as trip_date,
    COUNT(*) as daily_trips,
    ROUND(SUM(fare_amount), 2) as daily_revenue,
    COUNT(DISTINCT driver_id) as active_drivers,
    COUNT(DISTINCT rider_id) as active_riders,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) * 100.0 / COUNT(*) as completion_rate
  FROM workspace.default.uber_trips
  WHERE DATE(pickup_time) >= CURRENT_DATE() - INTERVAL 7 DAY
  GROUP BY DATE(pickup_time)
)
SELECT 
  trip_date,
  daily_trips,
  daily_revenue,
  active_drivers,
  active_riders,
  avg_fare,
  ROUND(completion_rate, 2) as completion_rate_pct,
  ROUND(AVG(daily_revenue) OVER(ORDER BY trip_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as rolling_3day_avg_revenue
FROM last_7_days
ORDER BY trip_date DESC;

-- Business Insight:
-- Monitor weekly trends and identify outlier days
-- Use rolling averages to smooth out daily volatility

trip_date,daily_trips,daily_revenue,active_drivers,active_riders,avg_fare,completion_rate_pct,rolling_3day_avg_revenue


In [0]:
%sql
-- Comprehensive KPI Dashboard - Single View
-- Demonstrates: Multiple aggregations, Business KPIs

WITH overall_metrics AS (
  SELECT 
    COUNT(*) as total_trips,
    COUNT(DISTINCT driver_id) as total_drivers,
    COUNT(DISTINCT rider_id) as total_riders,
    COUNT(DISTINCT city) as cities_operational,
    ROUND(SUM(fare_amount), 2) as total_revenue,
    ROUND(AVG(fare_amount), 2) as avg_fare,
    ROUND(SUM(distance_km), 2) as total_distance_covered,
    COUNT(CASE WHEN status = 'Completed' THEN 1 END) as completed_trips,
    COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) as cancelled_trips,
    COUNT(CASE WHEN status = 'No-Show' THEN 1 END) as no_show_trips
  FROM workspace.default.uber_trips
)
SELECT 
  total_trips,
  completed_trips,
  ROUND(completed_trips * 100.0 / total_trips, 2) as completion_rate_pct,
  total_drivers,
  total_riders,
  ROUND(total_trips * 1.0 / total_drivers, 2) as trips_per_driver,
  ROUND(total_trips * 1.0 / total_riders, 2) as trips_per_rider,
  total_revenue,
  avg_fare,
  ROUND(total_revenue / total_trips, 2) as revenue_per_trip,
  ROUND(total_revenue / total_drivers, 2) as revenue_per_driver,
  total_distance_covered,
  cities_operational,
  cancelled_trips,
  no_show_trips,
  ROUND(cancelled_trips * 100.0 / total_trips, 2) as cancellation_rate_pct,
  ROUND(no_show_trips * 100.0 / total_trips, 2) as no_show_rate_pct
FROM overall_metrics;

-- Business Insight:
-- Share this KPI snapshot in weekly leadership meetings
-- Track month-over-month changes for each metric

total_trips,completed_trips,completion_rate_pct,total_drivers,total_riders,trips_per_driver,trips_per_rider,total_revenue,avg_fare,revenue_per_trip,revenue_per_driver,total_distance_covered,cities_operational,cancelled_trips,no_show_trips,cancellation_rate_pct,no_show_rate_pct
50000,42540,85.08,8963,38352,5.58,1.30,798810.27,15.98,15.98,89.12,350403.51,6,4984,2476,9.97,4.95
